# Análise Exploratória — Gold

Este notebook valida os dados da camada Gold após a execução do `etl_gold.py`.

O objetivo é confirmar que:
- As 3 tabelas Gold foram criadas corretamente
- Os filtros de negócio foram aplicados
- O volume e distribuição dos dados estão corretos

As respostas das queries do case estão em `analysis/queries.sql`.

**Database:** `ifood_case_gold`  
**Tabelas:**
- `table_all_taxi_gold` — dados consolidados Yellow + Green
- `table_avg_total_amount_gold` — resposta Query 1
- `table_avg_passengers_gold` — resposta Query 2

## 1. Importações e configurações

In [1]:
import logging
import warnings
import awswrangler as wr
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning, module="awswrangler")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] exploratory_gold - %(message)s",
    datefmt="%d-%m-%Y %H:%M:%S",
)
logger = logging.getLogger("exploratory_gold")

logging.getLogger("boto3").setLevel(logging.WARNING)
logging.getLogger("botocore").setLevel(logging.WARNING)
logging.getLogger("awswrangler").setLevel(logging.WARNING)
logging.getLogger("urllib3").setLevel(logging.WARNING)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

DATABASE  = "ifood_case_gold"
S3_OUTPUT = "s3://ifood-case-data-lake-kaique/athena-results/"

logger.info("Bibliotecas carregadas com sucesso")

08-06-2026 08:32:13 [INFO] exploratory_gold - Bibliotecas carregadas com sucesso


## 2. Verificação de disponibilidade das tabelas

Verificamos se as 3 tabelas Gold foram criadas corretamente no Glue Catalog.

In [2]:
logger.info("Verificando tabelas no Glue Catalog | database=%s", DATABASE)

tabelas_catalog = wr.catalog.get_tables(database=DATABASE)
df_catalog = pd.DataFrame(tabelas_catalog)[["Name", "DatabaseName", "TableType"]]

logger.info("Tabelas encontradas | total=%s", len(df_catalog))
display(
    df_catalog.style
    .set_caption("Tabelas no Glue Catalog — ifood_case_gold")
    .set_properties(**{"text-align": "left"})
)

08-06-2026 08:32:14 [INFO] exploratory_gold - Verificando tabelas no Glue Catalog | database=ifood_case_gold
08-06-2026 08:32:16 [INFO] exploratory_gold - Tabelas encontradas | total=3


,Name,DatabaseName,TableType
0,table_all_taxi_gold,ifood_case_gold,EXTERNAL_TABLE
1,table_avg_passengers_gold,ifood_case_gold,EXTERNAL_TABLE
2,table_avg_total_amount_gold,ifood_case_gold,EXTERNAL_TABLE


## 3. Validação da table_all_taxi_gold

Verificamos o volume e distribuição dos dados consolidados.

In [3]:
logger.info("Validando table_all_taxi_gold")

df = wr.athena.read_sql_query(
    sql="""
        SELECT
            taxi_type,
            partition_month as mes,
            COUNT(*) as total_corridas,
            ROUND(AVG(total_amount), 2) as avg_total_amount,
            ROUND(AVG(passenger_count), 2) as avg_passenger_count,
            MIN(passenger_count) as min_passenger,
            MAX(passenger_count) as max_passenger,
            MIN(total_amount) as min_total_amount,
            MAX(total_amount) as max_total_amount
        FROM table_all_taxi_gold
        GROUP BY taxi_type, partition_month
        ORDER BY taxi_type, partition_month
    """,
    database=DATABASE,
    s3_output=S3_OUTPUT,
)

logger.info("Validacao concluida")
display(
    df.set_index(["taxi_type", "mes"])
    .style
    .set_caption("Resumo table_all_taxi_gold por Taxi e Mes")
    .format("{:,}", subset=["total_corridas"])
    .format("{:.2f}", subset=["avg_total_amount", "avg_passenger_count",
                              "min_total_amount", "max_total_amount"])
    .set_properties(**{"text-align": "right"})
)

08-06-2026 08:32:22 [INFO] exploratory_gold - Validando table_all_taxi_gold
08-06-2026 08:32:32 [INFO] exploratory_gold - Validacao concluida


## 4. Validação dos filtros aplicados

Confirmamos que os filtros de negócio foram aplicados corretamente.

In [4]:
logger.info("Validando filtros aplicados")

df = wr.athena.read_sql_query(
    sql="""
        SELECT
            SUM(CASE WHEN total_amount <= 0 THEN 1 ELSE 0 END) as total_amount_invalido,
            SUM(CASE WHEN passenger_count <= 0 THEN 1 ELSE 0 END) as passenger_count_zero,
            SUM(CASE WHEN passenger_count > 6 THEN 1 ELSE 0 END) as passenger_count_acima_6,
            SUM(CASE WHEN YEAR(tpep_pickup_datetime) != 2023 THEN 1 ELSE 0 END) as ano_invalido,
            SUM(CASE WHEN MONTH(tpep_pickup_datetime) NOT BETWEEN 1 AND 5
                THEN 1 ELSE 0 END) as mes_invalido,
            COUNT(*) as total
        FROM table_all_taxi_gold
    """,
    database=DATABASE,
    s3_output=S3_OUTPUT,
)

logger.info("Validacao de filtros concluida")
display(
    df.style
    .set_caption("Validacao dos Filtros — table_all_taxi_gold")
    .format("{:,}")
    .set_properties(**{"text-align": "right"})
)

08-06-2026 08:32:32 [INFO] exploratory_gold - Validando filtros aplicados
08-06-2026 08:32:43 [INFO] exploratory_gold - Validacao de filtros concluida


,total_amount_invalido,passenger_count_zero,passenger_count_acima_6,ano_invalido,mes_invalido,total
0,0,0,0,0,0,"15,653,279"


## 5. Amostra dos dados

Verificamos visualmente os dados consolidados na camada Gold.

In [5]:
logger.info("Carregando amostra dos dados")

df = wr.athena.read_sql_query(
    sql="""
        SELECT *
        FROM table_all_taxi_gold
        WHERE partition_year = 2023
        AND   partition_month = 1
        LIMIT 5
    """,
    database=DATABASE,
    s3_output=S3_OUTPUT,
)

logger.info("Amostra carregada | linhas=%s", len(df))
display(
    df.style
    .set_caption("Amostra — table_all_taxi_gold (Janeiro 2023)")
    .set_properties(**{"text-align": "left"})
)

08-06-2026 08:32:59 [INFO] exploratory_gold - Carregando amostra dos dados
08-06-2026 08:33:10 [INFO] exploratory_gold - Amostra carregada | linhas=5


,vendorid,passenger_count,total_amount,tpep_pickup_datetime,tpep_dropoff_datetime,taxi_type,partition_year,partition_month
0,2,1,14.300000,2023-01-01 00:32:10,2023-01-01 00:40:36,yellow,2023,1
1,2,1,16.900000,2023-01-01 00:55:08,2023-01-01 01:01:27,yellow,2023,1
2,2,1,34.900000,2023-01-01 00:25:04,2023-01-01 00:37:49,yellow,2023,1
3,2,1,19.680000,2023-01-01 00:10:29,2023-01-01 00:21:19,yellow,2023,1
4,2,1,27.800000,2023-01-01 00:50:34,2023-01-01 01:02:52,yellow,2023,1


## 6. Conclusões

Com base na validação da camada Gold:

| Filtro | Justificativa |
|---|---|
| `total_amount > 0` | Valores negativos são estornos — não representam corridas reais |
| `passenger_count > 0` | Corridas sem passageiro registrado são inválidas |
| `passenger_count <= 6` | Máximo físico permitido pela TLC em táxis NYC |
| `YEAR(pickup) = 2023` | Remove datas erradas de 2001, 2008 etc |
| `MONTH(pickup) entre 1 e 5` | Limita ao período Jan-Mai 2023 do enunciado |

As respostas das queries estão em `analysis/queries.sql`.